#  BALANCEO 

**IMPORTANTE**: Recuerda hacer una copia de esta plantilla para no machacar la original.

## IMPORT PACKAGES

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
%matplotlib inline

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import TomekLinks
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import roc_auc_score

# Autocomplete settings
%config IPCompleter.greedy=True

## IMPORT DATA

In [2]:
project_path = Path('/Users/gloriadelriomarquez/Documents/Career/Projects/ml-lab/client_churn')
data_path = project_path / "02_data" / "train"

X_file_name = "X_selected.pickle"
y_file_name = "y_selected.pickle"

X = pd.read_pickle(data_path / X_file_name)
y = pd.read_pickle(data_path / y_file_name)

## Check target distribution (class imbalance)

In [3]:
y.value_counts(normalize=True)

is_churn
0    0.91011
1    0.08989
Name: proportion, dtype: Float64

## Class balancing

We will evaluate whether applying class balancing improves model performance.  
Several balancing methods will be tested, including the option of leaving the data unbalanced.

### No balancing (baseline)

In [4]:
# Create train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

# Instantiate the model (no balancing)
lr_no_balancing = LogisticRegression( 
    solver="lbfgs",
    l1_ratio=0,       # Regularización L2
    C=1.0,
    max_iter=2000
)

# Train
lr_no_balancing.fit(X_train, y_train)

# Apply on original test set
pred_lr_no_balancing = lr_no_balancing.predict_proba(X_test)[:, 1]

# Evaluate
roc_lr_no_balancing = roc_auc_score(y_test, pred_lr_no_balancing)

roc_lr_no_balancing

0.854160601575475

### Regressionwith class_weight

In [11]:
# Logistic Regression with class weights
lr_class_weight = LogisticRegression(
    solver="lbfgs",
    l1_ratio=0,              # Regularización L2
    C=1.0,
    max_iter=2000,
    class_weight="balanced"
)

# Train
lr_class_weight.fit(X_train, y_train)

# Apply on original test set
pred_lr_class_weight = lr_class_weight.predict_proba(X_test)[:, 1]

# Evaluate
roc_lr_class_weight = roc_auc_score(y_test, pred_lr_class_weight)

roc_lr_class_weight

0.8622059074306142

### Create balanced dataset using undersampling

✔️ Balancear SOLO el train  
✔️ Evaluar SIEMPRE en el test original (test_X, test_y)  

In [5]:
# Instantiate the undersampler
rus = RandomUnderSampler(sampling_strategy=1)

# Apply undersampling ONLY on the training data
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

# Instantiate the model
lr_rus = LogisticRegression( 
    solver="lbfgs",
    l1_ratio=0,       # Regularización L2
    C=1.0,
    max_iter=2000
)

# Train
lr_rus.fit(X_train_rus, y_train_rus)

# Predict on ORIGINAL test set
pred_lr_rus = lr_rus.predict_proba(X_test)[:, 1]

# Evaluate on ORIGINAL test labels
roc_lr_rus = roc_auc_score(y_test, pred_lr_rus)

roc_lr_rus

0.8622233088139415

### Create balanced dataset using oversampling

In [6]:
# Instantiate the oversampler
ros = RandomOverSampler(sampling_strategy=1)

# Apply oversampling ONLY on the training set
X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)

# Instantiate the model
lr_ros = LogisticRegression( 
    solver="lbfgs",
    l1_ratio=0,       # Regularización L2
    C=1.0,
    max_iter=2000
)

# Train
lr_ros.fit(X_train_ros, y_train_ros)

# Apply on ORIGINAL (unbalanced) test set
pred_lr_ros = lr_ros.predict_proba(X_test)[:, 1]

# Evaluate on ORIGINAL labels
roc_lr_ros = roc_auc_score(y_test, pred_lr_ros)

roc_lr_ros

0.8621203486080096

### Create balanced dataset using SMOTE-Tomek

In [7]:
# Instantiate TomekLinks and SMOTE
tom = TomekLinks()
smo = SMOTE(sampling_strategy=1)

# Instantiate SMOTE-Tomek
sto = SMOTETomek(
    sampling_strategy=1,
    smote=smo,
    tomek=tom
)

# Apply SMOTE-Tomek ONLY on the training set
X_train_sto, y_train_sto = sto.fit_resample(X_train, y_train)

# Instantiate the model
lr_sto = LogisticRegression( 
    solver="lbfgs",
    l1_ratio=0,       # Regularización L2
    C=1.0,
    max_iter=2000
)

# Train
lr_sto.fit(X_train_sto, y_train_sto)

# Apply on ORIGINAL (unbalanced) test set
pred_lr_sto = lr_sto.predict_proba(X_test)[:, 1]

# Evaluate on ORIGINAL labels
roc_lr_sto = roc_auc_score(y_test, pred_lr_sto)

roc_lr_sto

0.8603313684494154

### Evaluate the best balancing strategy – results

In [15]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Probabilidades de cada estrategia
results_by_method = {
    "No balancing": pred_lr_no_balancing,
    "Class Weight Balanced" : pred_lr_class_weight,
    "Random UnderSampling": pred_lr_rus,
    "Random OverSampling": pred_lr_ros,
    "SMOTE - Tomek": pred_lr_sto
}

def compute_metrics_df(y_true, probabilities, method, threshold=0.5):

    # Convertir probabilidades a predicciones binarias
    y_pred = (np.asarray(probabilities) >= threshold).astype(int)

    # Matriz de confusión
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    return {
        "method": method,
        "AUC": roc_auc_score(y_true, probabilities),
        "AP": average_precision_score(y_true, probabilities),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }


# Comparar estrategias con umbral 0.5
df_metrics = pd.DataFrame([
    compute_metrics_df(
        y_true=y_test,
        probabilities=probs,
        method=method,
        threshold=0.5
    )
    for method, probs in results_by_method.items()
])

display(df_metrics.round(4))


,method,AUC,AP,Accuracy,Precision,Recall,F1,TN,FP,FN,TP
0,No balancing,0.8542,0.5973,0.9345,0.8152,0.3515,0.4912,184113,1460,11887,6442
1,Class Weight Balanced,0.8622,0.5868,0.8772,0.4017,0.7477,0.5226,165162,20411,4624,13705
2,Random UnderSampling,0.8622,0.5869,0.8759,0.3986,0.7479,0.5200,164891,20682,4621,13708
3,Random OverSampling,0.8621,0.5869,0.8770,0.4011,0.7478,0.5222,165110,20463,4623,13706
4,SMOTE - Tomek,0.8603,0.5867,0.8767,0.4004,0.7466,0.5213,165081,20492,4644,13685


### Select the final balancing method
Uncomment the chosen method and leave the rest commented

In [7]:
# Final balancing strategy selected after model comparison
best_balancing_method = "No balancing"

# Methods tested:
# "No balancing"
# "Random UnderSampling"
# "Random OverSampling"
# "SMOTE-Tomek"

